In [ ]:
# ============================================================
# PT -> PA Cross-Physics Mapping
# U-shaped Neural Operator (U-NO)
#
# Input:
#     PT [B, 1, 501, 200]
#
# Output:
#     PA [B, 1, 501, 200]
#
# Architecture:
#
# Input lifting
#   ->
# Encoder Neural Operator blocks
#   ->
# Bottleneck operator
#   ->
# Decoder Neural Operator blocks
#   + skip connections
#   ->
# Projection
#
# ============================================================


# ============================================================
# Imports
# ============================================================

import os
import csv
import glob
import random

from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import h5py

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader


# ============================================================
# Config
# ============================================================

DATA_ROOT = "Training dataset"


RUN_TIME = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUT_DIR = os.path.join(
    "uno2d_mat_dataset_results",
    f"run_{RUN_TIME}"
)


os.makedirs(
    OUT_DIR,
    exist_ok=True
)


print("Results will be saved to:")
print(OUT_DIR)


# ============================================================
# Dataset split
# ============================================================

TRAIN_RATIO = 0.8

VAL_RATIO = 0.1

TEST_RATIO = 0.1

SEED = 42


# ============================================================
# Device
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("Using device:", DEVICE)


# ============================================================
# Data size
# ============================================================

NT = 501

NX = 200


# ============================================================
# Training parameters
# ============================================================

EPOCHS = 1000

BATCH_SIZE = 4

LR = 1e-3

WEIGHT_DECAY = 1e-5


# ============================================================
# U-NO parameters
# ============================================================

IN_CHANNELS = 1

OUT_CHANNELS = 1


# base number of feature channels
BASE_WIDTH = 32


# Fourier modes at each scale
#
# These will automatically be clipped if feature map
# is smaller than the requested number.
#
MODES_T_LEVEL1 = 64
MODES_X_LEVEL1 = 40

MODES_T_LEVEL2 = 32
MODES_X_LEVEL2 = 24

MODES_T_LEVEL3 = 16
MODES_X_LEVEL3 = 12

MODES_T_BOTTLENECK = 8
MODES_X_BOTTLENECK = 6


# ============================================================
# Random seed
# ============================================================

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)


if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED
    )


# ============================================================
# Find all MAT files
# ============================================================

all_files = glob.glob(

    os.path.join(
        DATA_ROOT,
        "**",
        "*.mat"
    ),

    recursive=True
)


all_files = sorted(
    all_files
)


print(
    "Total .mat samples found:",
    len(all_files)
)


assert len(all_files) > 0, \
    f"No .mat files found under: {DATA_ROOT}"


if len(all_files) != 1000:

    print(
        f"Warning: expected 1000 samples, "
        f"but found {len(all_files)}"
    )


# ============================================================
# Random split
# ============================================================

random.shuffle(
    all_files
)


n_total = len(
    all_files
)


n_train = int(
    n_total * TRAIN_RATIO
)


n_val = int(
    n_total * VAL_RATIO
)


train_files = all_files[
    :n_train
]


val_files = all_files[
    n_train:
    n_train + n_val
]


test_files = all_files[
    n_train + n_val:
]


print("\nDataset split:")


print(
    "Train samples:",
    len(train_files)
)


print(
    "Val samples  :",
    len(val_files)
)


print(
    "Test samples :",
    len(test_files)
)


# ============================================================
# Compute normalization statistics
#
# TRAINING DATA ONLY
# ============================================================

def compute_statistics(
    file_list
):

    pt_sum = 0.0
    pt_sq_sum = 0.0

    pa_sum = 0.0
    pa_sq_sum = 0.0

    n_elements = 0


    print(
        "\nCalculating normalization statistics..."
    )


    for mat_path in tqdm(
        file_list,
        desc="Statistics",
        ncols=120
    ):


        with h5py.File(
            mat_path,
            "r"
        ) as f:


            PT = np.asarray(
                f["PT"],
                dtype=np.float64
            )


            PA = np.asarray(
                f["PA"],
                dtype=np.float64
            )


        # ====================================================
        # Shape
        # ====================================================

        assert PT.shape == (NT, NX), \
            f"Wrong PT shape in {mat_path}: {PT.shape}"


        assert PA.shape == (NT, NX), \
            f"Wrong PA shape in {mat_path}: {PA.shape}"


        pt_sum += PT.sum()


        pt_sq_sum += np.square(
            PT
        ).sum()


        pa_sum += PA.sum()


        pa_sq_sum += np.square(
            PA
        ).sum()


        n_elements += PT.size


    # ========================================================
    # Mean
    # ========================================================

    pt_mean = (
        pt_sum
        /
        n_elements
    )


    pa_mean = (
        pa_sum
        /
        n_elements
    )


    # ========================================================
    # Variance
    # ========================================================

    pt_var = (
        pt_sq_sum
        /
        n_elements
        -
        pt_mean ** 2
    )


    pa_var = (
        pa_sq_sum
        /
        n_elements
        -
        pa_mean ** 2
    )


    pt_var = max(
        pt_var,
        0.0
    )


    pa_var = max(
        pa_var,
        0.0
    )


    # ========================================================
    # Standard deviation
    # ========================================================

    pt_std = (
        np.sqrt(
            pt_var
        )
        +
        1e-8
    )


    pa_std = (
        np.sqrt(
            pa_var
        )
        +
        1e-8
    )


    return (
        pt_mean,
        pt_std,
        pa_mean,
        pa_std
    )


# ============================================================
# Normalization statistics
# ============================================================

(
    pt_mean,
    pt_std,
    pa_mean,
    pa_std

) = compute_statistics(
    train_files
)


print(
    "\nNormalization statistics:"
)


print(
    f"PT mean = {pt_mean:.6e}"
)


print(
    f"PT std  = {pt_std:.6e}"
)


print(
    f"PA mean = {pa_mean:.6e}"
)


print(
    f"PA std  = {pa_std:.6e}"
)


# ============================================================
# Save normalization
# ============================================================

np.savez(

    os.path.join(
        OUT_DIR,
        "normalization_parameters.npz"
    ),

    pt_mean=pt_mean,
    pt_std=pt_std,
    pa_mean=pa_mean,
    pa_std=pa_std
)


# ============================================================
# Dataset
# ============================================================

class MatHeatAcousticDataset(Dataset):


    def __init__(
        self,
        file_list,
        pt_mean,
        pt_std,
        pa_mean,
        pa_std
    ):


        self.file_list = file_list

        self.pt_mean = pt_mean
        self.pt_std = pt_std

        self.pa_mean = pa_mean
        self.pa_std = pa_std


    def __len__(
        self
    ):

        return len(
            self.file_list
        )


    def __getitem__(
        self,
        idx
    ):


        mat_path = self.file_list[
            idx
        ]


        with h5py.File(
            mat_path,
            "r"
        ) as f:


            PT = np.asarray(
                f["PT"],
                dtype=np.float32
            )


            PA = np.asarray(
                f["PA"],
                dtype=np.float32
            )


        # ====================================================
        # Shape
        # ====================================================

        assert PT.shape == (NT, NX), \
            f"Wrong PT shape in {mat_path}: {PT.shape}"


        assert PA.shape == (NT, NX), \
            f"Wrong PA shape in {mat_path}: {PA.shape}"


        # ====================================================
        # Normalize
        # ====================================================

        PT = (
            PT
            -
            self.pt_mean
        ) / self.pt_std


        PA = (
            PA
            -
            self.pa_mean
        ) / self.pa_std


        # ====================================================
        # [T, X]
        #
        # ->
        #
        # [C, T, X]
        # ====================================================

        PT = torch.tensor(
            PT,
            dtype=torch.float32
        ).unsqueeze(
            0
        )


        PA = torch.tensor(
            PA,
            dtype=torch.float32
        ).unsqueeze(
            0
        )


        return (
            PT,
            PA
        )


    # ========================================================
    # Denormalization
    # ========================================================

    def denormalize_pa(
        self,
        x
    ):

        return (
            x
            *
            self.pa_std
            +
            self.pa_mean
        )


    def denormalize_pt(
        self,
        x
    ):

        return (
            x
            *
            self.pt_std
            +
            self.pt_mean
        )


# ============================================================
# Dataset creation
# ============================================================

train_dataset = MatHeatAcousticDataset(

    train_files,

    pt_mean,
    pt_std,

    pa_mean,
    pa_std
)


val_dataset = MatHeatAcousticDataset(

    val_files,

    pt_mean,
    pt_std,

    pa_mean,
    pa_std
)


test_dataset = MatHeatAcousticDataset(

    test_files,

    pt_mean,
    pt_std,

    pa_mean,
    pa_std
)


# ============================================================
# DataLoaders
# ============================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


# ============================================================
# Dataset check
# ============================================================

PT_batch, PA_batch = next(
    iter(
        train_loader
    )
)


print(
    "\nBatch check:"
)


print(
    "PT batch shape:",
    PT_batch.shape
)


print(
    "PA batch shape:",
    PA_batch.shape
)


# ============================================================
# 2D Spectral Convolution
#
# Input:
#
# [B,C,T,X]
#
# ============================================================

class SpectralConv2d(nn.Module):


    def __init__(
        self,
        in_channels,
        out_channels,
        modes_t,
        modes_x
    ):


        super().__init__()


        self.in_channels = in_channels

        self.out_channels = out_channels


        self.modes_t = modes_t

        self.modes_x = modes_x


        scale = (
            1.0
            /
            (
                in_channels
                *
                out_channels
            )
        )


        # ====================================================
        # Positive temporal modes
        # ====================================================

        self.weights_pos = nn.Parameter(

            scale
            *
            torch.randn(

                in_channels,

                out_channels,

                modes_t,

                modes_x,

                dtype=torch.cfloat
            )
        )


        # ====================================================
        # Negative temporal modes
        # ====================================================

        self.weights_neg = nn.Parameter(

            scale
            *
            torch.randn(

                in_channels,

                out_channels,

                modes_t,

                modes_x,

                dtype=torch.cfloat
            )
        )


    # ========================================================
    # Complex multiplication
    # ========================================================

    def compl_mul2d(
        self,
        x,
        weights
    ):


        return torch.einsum(

            "bixy,ioxy->boxy",

            x,

            weights
        )


    # ========================================================
    # Forward
    # ========================================================

    def forward(
        self,
        x
    ):


        B, C, T, X = x.shape


        # ====================================================
        # Fourier transform
        # ====================================================

        x_ft = torch.fft.rfftn(

            x,

            dim=(-2, -1)
        )


        # ====================================================
        # Output spectrum
        # ====================================================

        out_ft = torch.zeros(

            B,

            self.out_channels,

            T,

            X // 2 + 1,

            dtype=torch.cfloat,

            device=x.device
        )


        # ====================================================
        # Safe number of modes
        # ====================================================

        mt = min(

            self.modes_t,

            T // 2
        )


        mx = min(

            self.modes_x,

            X // 2 + 1
        )


        # ====================================================
        # Positive temporal frequencies
        # ====================================================

        out_ft[
            :,
            :,
            :mt,
            :mx
        ] = self.compl_mul2d(

            x_ft[
                :,
                :,
                :mt,
                :mx
            ],

            self.weights_pos[
                :,
                :,
                :mt,
                :mx
            ]
        )


        # ====================================================
        # Negative temporal frequencies
        # ====================================================

        out_ft[
            :,
            :,
            -mt:,
            :mx
        ] = self.compl_mul2d(

            x_ft[
                :,
                :,
                -mt:,
                :mx
            ],

            self.weights_neg[
                :,
                :,
                :mt,
                :mx
            ]
        )


        # ====================================================
        # Inverse FFT
        # ====================================================

        x = torch.fft.irfftn(

            out_ft,

            s=(T, X),

            dim=(-2, -1)
        )


        return x


# ============================================================
# Neural Operator Block
#
# spectral branch
#
# +
#
# local convolution
#
# ============================================================

class OperatorBlock(nn.Module):


    def __init__(
        self,
        in_channels,
        out_channels,
        modes_t,
        modes_x
    ):


        super().__init__()


        # ====================================================
        # Spectral operator
        # ====================================================

        self.spectral = SpectralConv2d(

            in_channels,

            out_channels,

            modes_t,

            modes_x
        )


        # ====================================================
        # Local pointwise operator
        # ====================================================

        self.local = nn.Conv2d(

            in_channels,

            out_channels,

            kernel_size=1
        )


        # ====================================================
        # Normalization
        # ====================================================

        groups = min(
            8,
            out_channels
        )


        while (
            out_channels % groups != 0
            and
            groups > 1
        ):

            groups -= 1


        self.norm = nn.GroupNorm(

            groups,

            out_channels
        )


    def forward(
        self,
        x
    ):


        x1 = self.spectral(
            x
        )


        x2 = self.local(
            x
        )


        x = (
            x1
            +
            x2
        )


        x = self.norm(
            x
        )


        x = F.gelu(
            x
        )


        return x


# ============================================================
# Downsampling operator block
#
# Operator mapping first,
# then resolution reduction.
# ============================================================

class DownOperatorBlock(nn.Module):


    def __init__(
        self,
        in_channels,
        out_channels,
        modes_t,
        modes_x
    ):


        super().__init__()


        self.operator = OperatorBlock(

            in_channels,

            out_channels,

            modes_t,

            modes_x
        )


    def forward(
        self,
        x
    ):


        # ====================================================
        # Operator transform
        # ====================================================

        skip = self.operator(
            x
        )


        # ====================================================
        # Downsampling
        #
        # ceil_mode=True helps preserve odd T=501
        # ====================================================

        x = F.avg_pool2d(

            skip,

            kernel_size=2,

            stride=2,

            ceil_mode=True
        )


        return (
            x,
            skip
        )


# ============================================================
# Up operator block
#
# Upsample
# +
# concatenate skip
# +
# operator transform
# ============================================================

class UpOperatorBlock(nn.Module):


    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
        modes_t,
        modes_x
    ):


        super().__init__()


        # ====================================================
        # Reduce decoder channels
        # ====================================================

        self.reduce = nn.Conv2d(

            in_channels,

            out_channels,

            kernel_size=1
        )


        # ====================================================
        # After concat:
        #
        # decoder channels
        # +
        # skip channels
        # ====================================================

        self.operator = OperatorBlock(

            out_channels
            +
            skip_channels,

            out_channels,

            modes_t,

            modes_x
        )


    def forward(
        self,
        x,
        skip
    ):


        # ====================================================
        # Exact upsample to skip size
        #
        # Important because T=501 is odd.
        # ====================================================

        x = F.interpolate(

            x,

            size=skip.shape[-2:],

            mode="bilinear",

            align_corners=False
        )


        # ====================================================
        # Channel reduction
        # ====================================================

        x = self.reduce(
            x
        )


        # ====================================================
        # Skip connection
        # ====================================================

        x = torch.cat(

            [
                x,
                skip
            ],

            dim=1
        )


        # ====================================================
        # Neural operator
        # ====================================================

        x = self.operator(
            x
        )


        return x


# ============================================================
# U-shaped Neural Operator
#
# PT -> PA
# ============================================================

class UNO2D(nn.Module):


    def __init__(
        self,
        in_channels=1,
        out_channels=1,
        base_width=32
    ):


        super().__init__()


        # ====================================================
        # Channels
        # ====================================================

        c1 = base_width

        c2 = base_width * 2

        c3 = base_width * 4

        c4 = base_width * 8


        # ====================================================
        # Input lifting
        #
        # input:
        #
        # PT + t-grid + x-grid
        # ====================================================

        self.lifting = nn.Conv2d(

            in_channels + 2,

            c1,

            kernel_size=1
        )


        # ====================================================
        # Encoder
        # ====================================================

        self.down1 = DownOperatorBlock(

            c1,

            c1,

            MODES_T_LEVEL1,

            MODES_X_LEVEL1
        )


        self.down2 = DownOperatorBlock(

            c1,

            c2,

            MODES_T_LEVEL2,

            MODES_X_LEVEL2
        )


        self.down3 = DownOperatorBlock(

            c2,

            c3,

            MODES_T_LEVEL3,

            MODES_X_LEVEL3
        )


        # ====================================================
        # Bottleneck
        # ====================================================

        self.bottleneck = OperatorBlock(

            c3,

            c4,

            MODES_T_BOTTLENECK,

            MODES_X_BOTTLENECK
        )


        # ====================================================
        # Decoder
        # ====================================================

        self.up3 = UpOperatorBlock(

            c4,

            c3,

            c3,

            MODES_T_LEVEL3,

            MODES_X_LEVEL3
        )


        self.up2 = UpOperatorBlock(

            c3,

            c2,

            c2,

            MODES_T_LEVEL2,

            MODES_X_LEVEL2
        )


        self.up1 = UpOperatorBlock(

            c2,

            c1,

            c1,

            MODES_T_LEVEL1,

            MODES_X_LEVEL1
        )


        # ====================================================
        # Final local refinement
        # ====================================================

        self.refine = nn.Sequential(

            nn.Conv2d(

                c1,

                c1,

                kernel_size=3,

                padding=1
            ),

            nn.GELU(),

            nn.Conv2d(

                c1,

                c1,

                kernel_size=3,

                padding=1
            ),

            nn.GELU()
        )


        # ====================================================
        # Projection
        # ====================================================

        self.proj1 = nn.Conv2d(

            c1,

            128,

            kernel_size=1
        )


        self.proj2 = nn.Conv2d(

            128,

            out_channels,

            kernel_size=1
        )


    # ========================================================
    # Coordinate grid
    # ========================================================

    def get_grid(
        self,
        shape,
        device
    ):


        B, C, T, X = shape


        t = torch.linspace(

            0.0,

            1.0,

            T,

            device=device
        )


        x = torch.linspace(

            0.0,

            1.0,

            X,

            device=device
        )


        tt, xx = torch.meshgrid(

            t,

            x,

            indexing="ij"
        )


        grid = torch.stack(

            [
                tt,
                xx
            ],

            dim=0
        )


        grid = grid.unsqueeze(
            0
        )


        grid = grid.repeat(

            B,

            1,

            1,

            1
        )


        return grid


    # ========================================================
    # Forward
    # ========================================================

    def forward(
        self,
        x
    ):


        original_size = x.shape[
            -2:
        ]


        # ====================================================
        # Coordinate grid
        # ====================================================

        grid = self.get_grid(

            x.shape,

            x.device
        )


        # ====================================================
        # PT + t + x
        # ====================================================

        x = torch.cat(

            [
                x,
                grid
            ],

            dim=1
        )


        # ====================================================
        # Lifting
        # ====================================================

        x = self.lifting(
            x
        )


        # ====================================================
        # Encoder
        # ====================================================

        x, skip1 = self.down1(
            x
        )


        x, skip2 = self.down2(
            x
        )


        x, skip3 = self.down3(
            x
        )


        # ====================================================
        # Bottleneck
        # ====================================================

        x = self.bottleneck(
            x
        )


        # ====================================================
        # Decoder
        # ====================================================

        x = self.up3(

            x,

            skip3
        )


        x = self.up2(

            x,

            skip2
        )


        x = self.up1(

            x,

            skip1
        )


        # ====================================================
        # Local refinement
        # ====================================================

        residual = x


        x = self.refine(
            x
        )


        x = (
            x
            +
            residual
        )


        # ====================================================
        # Projection
        # ====================================================

        x = F.gelu(

            self.proj1(
                x
            )
        )


        x = self.proj2(
            x
        )


        # ====================================================
        # Safety
        # ====================================================

        if (
            x.shape[-2:]
            !=
            original_size
        ):


            x = F.interpolate(

                x,

                size=original_size,

                mode="bilinear",

                align_corners=False
            )


        return x


# ============================================================
# Build model
# ============================================================

model = UNO2D(

    in_channels=IN_CHANNELS,

    out_channels=OUT_CHANNELS,

    base_width=BASE_WIDTH

).to(
    DEVICE
)


# ============================================================
# Model shape check
# ============================================================

print(
    "\nChecking model dimensions..."
)


with torch.no_grad():


    test_input = torch.randn(

        1,

        1,

        NT,

        NX,

        device=DEVICE
    )


    test_output = model(
        test_input
    )


print(
    "Input shape :",
    test_input.shape
)


print(
    "Output shape:",
    test_output.shape
)


assert (

    test_output.shape

    ==

    test_input.shape

), \
    f"Output shape mismatch: {test_output.shape}"


del test_input

del test_output


if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ============================================================
# Number of parameters
# ============================================================

n_params = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad
)


print(
    f"\nTrainable parameters: {n_params:,}"
)


# ============================================================
# Relative L2
# ============================================================

def relative_l2(
    pred,
    target
):


    numerator = torch.norm(

        pred
        -
        target
    )


    denominator = (

        torch.norm(
            target
        )

        +

        1e-8
    )


    return (

        numerator
        /
        denominator
    )


# ============================================================
# Train one epoch
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer
):


    model.train()


    total_mse = 0.0

    total_rel = 0.0


    for heat, acoustic in loader:


        heat = heat.to(

            DEVICE,

            non_blocking=True
        )


        acoustic = acoustic.to(

            DEVICE,

            non_blocking=True
        )


        # ====================================================
        # Forward
        # ====================================================

        pred = model(
            heat
        )


        # ====================================================
        # Loss
        # ====================================================

        mse = F.mse_loss(

            pred,

            acoustic
        )


        rel = relative_l2(

            pred,

            acoustic
        )


        loss = (

            mse

            +

            0.1
            *
            rel
        )


        # ====================================================
        # Backpropagation
        # ====================================================

        optimizer.zero_grad(
            set_to_none=True
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            max_norm=1.0
        )


        optimizer.step()


        # ====================================================
        # Statistics
        # ====================================================

        total_mse += mse.item()

        total_rel += rel.item()


    return (

        total_mse
        /
        len(loader),

        total_rel
        /
        len(loader)
    )


# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def evaluate(
    model,
    loader
):


    model.eval()


    total_mse = 0.0

    total_mae = 0.0

    total_rel = 0.0


    for heat, acoustic in loader:


        heat = heat.to(

            DEVICE,

            non_blocking=True
        )


        acoustic = acoustic.to(

            DEVICE,

            non_blocking=True
        )


        pred = model(
            heat
        )


        mse = F.mse_loss(

            pred,

            acoustic
        )


        mae = F.l1_loss(

            pred,

            acoustic
        )


        rel = relative_l2(

            pred,

            acoustic
        )


        total_mse += mse.item()

        total_mae += mae.item()

        total_rel += rel.item()


    return (

        total_mse
        /
        len(loader),

        total_mae
        /
        len(loader),

        total_rel
        /
        len(loader)
    )


# ============================================================
# Plot training curves
# ============================================================

def plot_loss(
    log_path
):


    data = np.loadtxt(

        log_path,

        delimiter=",",

        skiprows=1
    )


    if data.ndim == 1:

        data = data[
            None,
            :
        ]


    epoch = data[
        :,
        0
    ]


    train_mse = data[
        :,
        1
    ]


    train_rel = data[
        :,
        2
    ]


    val_mse = data[
        :,
        3
    ]


    val_rel = data[
        :,
        5
    ]


    # ========================================================
    # MSE
    # ========================================================

    plt.figure(
        figsize=(5, 3.8)
    )


    plt.semilogy(

        epoch,

        train_mse,

        label="Train MSE"
    )


    plt.semilogy(

        epoch,

        val_mse,

        label="Validation MSE"
    )


    plt.xlabel(
        "Epoch"
    )


    plt.ylabel(
        "MSE"
    )


    plt.legend(
        frameon=False
    )


    plt.tight_layout()


    plt.savefig(

        os.path.join(
            OUT_DIR,
            "loss_curve.png"
        ),

        dpi=300
    )


    plt.close()


    # ========================================================
    # Relative L2
    # ========================================================

    plt.figure(
        figsize=(5, 3.8)
    )


    plt.semilogy(

        epoch,

        train_rel,

        label="Train Rel. L2"
    )


    plt.semilogy(

        epoch,

        val_rel,

        label="Validation Rel. L2"
    )


    plt.xlabel(
        "Epoch"
    )


    plt.ylabel(
        "Relative L2"
    )


    plt.legend(
        frameon=False
    )


    plt.tight_layout()


    plt.savefig(

        os.path.join(
            OUT_DIR,
            "relative_l2_curve.png"
        ),

        dpi=300
    )


    plt.close()


# ============================================================
# Plot prediction
# ============================================================

@torch.no_grad()
def plot_prediction(
    model,
    dataset,
    sample_index=0
):


    model.eval()


    PT, PA = dataset[
        sample_index
    ]


    # ========================================================
    # Prediction
    # ========================================================

    PT_gpu = PT.unsqueeze(
        0
    ).to(
        DEVICE
    )


    pred = model(
        PT_gpu
    )


    pred = (

        pred
        .cpu()
        .squeeze(0)
        .squeeze(0)
        .numpy()
    )


    PT = (

        PT
        .squeeze(0)
        .numpy()
    )


    PA = (

        PA
        .squeeze(0)
        .numpy()
    )


    # ========================================================
    # Denormalize
    # ========================================================

    PT_real = dataset.denormalize_pt(
        PT
    )


    PA_real = dataset.denormalize_pa(
        PA
    )


    pred_real = dataset.denormalize_pa(
        pred
    )


    # ========================================================
    # Error
    # ========================================================

    err = (

        pred_real
        -
        PA_real
    )


    # ========================================================
    # Plot scale
    # ========================================================

    vmax = np.max(

        np.abs(
            PA_real
        )
    )


    vmax = max(
        vmax,
        1e-12
    )


    evmax = np.max(

        np.abs(
            err
        )
    )


    evmax = max(
        evmax,
        1e-12
    )


    # ========================================================
    # Figure
    # ========================================================

    plt.figure(
        figsize=(12, 3)
    )


    # --------------------------------------------------------
    # Input PT
    # --------------------------------------------------------

    plt.subplot(
        1,
        4,
        1
    )


    plt.imshow(

        PT_real,

        aspect="auto",

        cmap="inferno"
    )


    plt.title(
        "Input PT"
    )


    plt.xlabel(
        "x"
    )


    plt.ylabel(
        "t"
    )


    plt.colorbar()


    # --------------------------------------------------------
    # Ground truth PA
    # --------------------------------------------------------

    plt.subplot(
        1,
        4,
        2
    )


    plt.imshow(

        PA_real,

        aspect="auto",

        cmap="seismic",

        vmin=-vmax,

        vmax=vmax
    )


    plt.title(
        "GT PA"
    )


    plt.xlabel(
        "x"
    )


    plt.ylabel(
        "t"
    )


    plt.colorbar()


    # --------------------------------------------------------
    # Predicted PA
    # --------------------------------------------------------

    plt.subplot(
        1,
        4,
        3
    )


    plt.imshow(

        pred_real,

        aspect="auto",

        cmap="seismic",

        vmin=-vmax,

        vmax=vmax
    )


    plt.title(
        "Pred PA"
    )


    plt.xlabel(
        "x"
    )


    plt.ylabel(
        "t"
    )


    plt.colorbar()


    # --------------------------------------------------------
    # Error
    # --------------------------------------------------------

    plt.subplot(
        1,
        4,
        4
    )


    plt.imshow(

        err,

        aspect="auto",

        cmap="seismic",

        vmin=-evmax,

        vmax=evmax
    )


    plt.title(
        "Error"
    )


    plt.xlabel(
        "x"
    )


    plt.ylabel(
        "t"
    )


    plt.colorbar()


    plt.tight_layout()


    plt.savefig(

        os.path.join(
            OUT_DIR,
            "prediction_comparison.png"
        ),

        dpi=300,

        bbox_inches="tight"
    )


    plt.close()


    # ========================================================
    # Save numerical result
    # ========================================================

    np.savez(

        os.path.join(
            OUT_DIR,
            "prediction_result.npz"
        ),

        PT=PT_real,

        PA=PA_real,

        PA_pred=pred_real,

        error=err
    )


# ============================================================
# Optimizer
# ============================================================

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LR,

    weight_decay=WEIGHT_DECAY
)


# ============================================================
# Scheduler
# ============================================================

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(

    optimizer,

    T_max=EPOCHS
)


# ============================================================
# Training log
# ============================================================

log_path = os.path.join(

    OUT_DIR,

    "training_log.csv"
)


with open(

    log_path,

    "w",

    newline=""

) as f:


    writer = csv.writer(
        f
    )


    writer.writerow(

        [
            "epoch",
            "train_mse",
            "train_rel_l2",
            "val_mse",
            "val_mae",
            "val_rel_l2",
            "lr"
        ]
    )


# ============================================================
# Best model
# ============================================================

best_val = float(
    "inf"
)


best_model_path = os.path.join(

    OUT_DIR,

    "best_uno2d.pt"
)


# ============================================================
# Training loop
# ============================================================

epoch_bar = tqdm(

    range(
        1,
        EPOCHS + 1
    ),

    desc="Training",

    ncols=120
)


for epoch in epoch_bar:


    # ========================================================
    # Train
    # ========================================================

    train_mse, train_rel = train_one_epoch(

        model,

        train_loader,

        optimizer
    )


    # ========================================================
    # Validation
    # ========================================================

    (
        val_mse,
        val_mae,
        val_rel

    ) = evaluate(

        model,

        val_loader
    )


    # ========================================================
    # Learning rate
    # ========================================================

    scheduler.step()


    lr_now = optimizer.param_groups[
        0
    ]["lr"]


    # ========================================================
    # Log
    # ========================================================

    with open(

        log_path,

        "a",

        newline=""

    ) as f:


        writer = csv.writer(
            f
        )


        writer.writerow(

            [
                epoch,
                train_mse,
                train_rel,
                val_mse,
                val_mae,
                val_rel,
                lr_now
            ]
        )


    # ========================================================
    # Save best model
    # ========================================================

    if val_rel < best_val:


        best_val = val_rel


        torch.save(

            model.state_dict(),

            best_model_path
        )


    # ========================================================
    # Print
    # ========================================================

    if (
        epoch == 1
        or
        epoch % 10 == 0
    ):


        print(

            f"\nEpoch {epoch:04d} | "

            f"Train MSE {train_mse:.4e} | "

            f"Train Rel {train_rel:.4e} | "

            f"Val MSE {val_mse:.4e} | "

            f"Val MAE {val_mae:.4e} | "

            f"Val Rel {val_rel:.4e}"
        )


    epoch_bar.set_postfix(

        train_mse=f"{train_mse:.2e}",

        val_mse=f"{val_mse:.2e}",

        rel=f"{val_rel:.2e}",

        lr=f"{lr_now:.1e}"
    )


# ============================================================
# Load best model
# ============================================================

print(
    "\nLoading best model..."
)


model.load_state_dict(

    torch.load(

        best_model_path,

        map_location=DEVICE,

        weights_only=True
    )
)


# ============================================================
# Final test
# ============================================================

(
    test_mse,
    test_mae,
    test_rel

) = evaluate(

    model,

    test_loader
)


print(
    "\nFinal Test Results"
)


print(
    f"Test MSE     : {test_mse:.6e}"
)


print(
    f"Test MAE     : {test_mae:.6e}"
)


print(
    f"Test Rel L2  : {test_rel:.6e}"
)


# ============================================================
# Save results
# ============================================================

with open(

    os.path.join(
        OUT_DIR,
        "test_results.txt"
    ),

    "w"

) as f:


    f.write(
        "Final Test Results\n"
    )


    f.write(
        f"Test MSE     : "
        f"{test_mse:.6e}\n"
    )


    f.write(
        f"Test MAE     : "
        f"{test_mae:.6e}\n"
    )


    f.write(
        f"Test Rel L2  : "
        f"{test_rel:.6e}\n"
    )


# ============================================================
# Plot
# ============================================================

plot_loss(
    log_path
)


plot_prediction(

    model,

    test_dataset,

    sample_index=0
)


# ============================================================
# Finish
# ============================================================

print(
    f"\nBest validation Rel L2: "
    f"{best_val:.6e}"
)


print(
    f"Best model saved to: "
    f"{best_model_path}"
)


print(
    f"\nAll results saved to: "
    f"{OUT_DIR}"
)

Results will be saved to:
uno2d_mat_dataset_results/run_20260822_202604
Using device: cuda
Total .mat samples found: 800

Dataset split:
Train samples: 640
Val samples  : 80
Test samples : 80

Calculating normalization statistics...


Statistics:   0%|                                                                               | 0/640 [00:00…


Normalization statistics:
PT mean = 3.135567e+01
PT std  = 2.164740e+01
PA mean = -2.100887e+04
PA std  = 2.847824e+05

Batch check:
PT batch shape: torch.Size([4, 1, 501, 200])
PA batch shape: torch.Size([4, 1, 501, 200])

Checking model dimensions...
Input shape : torch.Size([1, 1, 501, 200])
Output shape: torch.Size([1, 1, 501, 200])

Trainable parameters: 50,487,009


Training:   0%|                                                                                | 0/1000 [00:00…